### Import required Python libraries for API handling and data processing

In [1]:
import requests
import pandas as pd
import sqlite3

### Define API endpoint and database connection

In [2]:
PDBe_API_URL = "https://www.ebi.ac.uk/pdbe/search/pdb/select"

QUERY = "enzyme"  

# SQLite database file
DB_NAME = "pdbe.db"
conn = sqlite3.connect(DB_NAME)

### This function fetches structural data from PDBe REST API using a search query.

In [3]:
def fetch_pdbe_data(query, rows=200):
    params = {
        "q": query,
        "wt": "json",
        "rows": rows
    }

    response = requests.get(PDBe_API_URL, params=params)

    if response.status_code != 200:
        raise Exception(f"API Error: {response.status_code}")

    data = response.json()
    docs = data.get("response", {}).get("docs", [])

    return docs


raw_data = fetch_pdbe_data(QUERY, rows=200)
print("Records fetched:", len(raw_data))

Records fetched: 200


### Convert nested JSON into structured tabular format using Pandas

In [4]:
def transform_data(docs):
    records = []

    for doc in docs:
        records.append({
            "pdb_id": doc.get("pdb_id"),
            "title": doc.get("title"),

            # REAL FIELD (sometimes missing)
            "resolution": doc.get("resolution"),

            # list → string fix
            "organism": (
                ", ".join(doc.get("organism_scientific_name"))
                if isinstance(doc.get("organism_scientific_name"), list)
                else doc.get("organism_scientific_name")
            )
        })

    return pd.DataFrame(records)


df = transform_data(raw_data)
df.head()

,pdb_id,title,resolution,organism
0,2awf,Structure of human Ubiquitin-conjugating enzym...,2.10,Homo sapiens
1,6d68,Ube2G1 in complex with ubiquitin variant Ubv.G1.1,2.36,Homo sapiens
2,7rzh,Insulin Degrading Enzyme O/O,3.80,Homo sapiens
3,7rzg,Insulin Degrading Enzyme O/pO,4.10,Homo sapiens
4,1gq2,Malic Enzyme from Pigeon Liver,2.50,Columba livia


### Clean missing values, remove duplicates, and standardize data types

In [6]:
df.isnull().sum()

pdb_id        0
title         0
resolution    5
organism      0
dtype: int64

In [7]:
def clean_data(df):
    df = df.drop_duplicates(subset=["pdb_id"])
    df = df.dropna(subset=["pdb_id"])

    df["resolution"] = pd.to_numeric(df["resolution"], errors="coerce")

    return df


df = clean_data(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 190 entries, 0 to 199
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   pdb_id      190 non-null    object 
 1   title       190 non-null    object 
 2   resolution  185 non-null    float64
 3   organism    190 non-null    object 
dtypes: float64(1), object(3)
memory usage: 7.4+ KB


In [9]:
df

,pdb_id,title,resolution,organism
0,2awf,Structure of human Ubiquitin-conjugating enzym...,2.100,Homo sapiens
1,6d68,Ube2G1 in complex with ubiquitin variant Ubv.G1.1,2.360,Homo sapiens
2,7rzh,Insulin Degrading Enzyme O/O,3.800,Homo sapiens
3,7rzg,Insulin Degrading Enzyme O/pO,4.100,Homo sapiens
4,1gq2,Malic Enzyme from Pigeon Liver,2.500,Columba livia
...,...,...,...,...
195,4auq,Structure of BIRC7-UbcH5b-Ub complex.,2.176,Homo sapiens
196,4a4c,Structure of phosphoTyr371-c-Cbl-UbcH5B-ZAP-70...,2.704,Homo sapiens
197,4a4b,Structure of modified phosphoTyr371-c-Cbl-UbcH...,2.789,Homo sapiens
198,4v3k,RNF38-UbcH5B-UB complex,2.040,Homo sapiens


### Define database schema for storing structural data

In [10]:
def create_table(conn):
    conn.execute("""
    CREATE TABLE IF NOT EXISTS pdbe_structures_pop (
        pdb_id TEXT PRIMARY KEY,
        title TEXT,
        resolution REAL,
        organism TEXT
    );
    """)
    conn.commit()

create_table(conn)

### Store transformed data into SQLite database

In [12]:
def load_data(df, conn):
    df = df.where(pd.notnull(df), None)  # NaN → None fix

    df.to_sql(
        "pdbe_structures_pop",
        conn,
        if_exists="append",
        index=False
    )

load_data(df, conn)
print("Data loaded successfully")

Data loaded successfully


### Improve query performance using database indexing

In [15]:
def create_indexes(conn):
    conn.execute("CREATE INDEX IF NOT EXISTS idx_pdb ON pdbe_structures_pop(pdb_id);")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_resolution ON pdbe_structures_pop(resolution);")
    conn.commit()

create_indexes(conn)

### Retrieve stored records from database

In [17]:
pd.read_sql("SELECT * FROM pdbe_structures_pop LIMIT 10;", conn)

,pdb_id,title,resolution,organism
0,2awf,Structure of human Ubiquitin-conjugating enzym...,2.10,Homo sapiens
1,6d68,Ube2G1 in complex with ubiquitin variant Ubv.G1.1,2.36,Homo sapiens
2,7rzh,Insulin Degrading Enzyme O/O,3.80,Homo sapiens
3,7rzg,Insulin Degrading Enzyme O/pO,4.10,Homo sapiens
4,1gq2,Malic Enzyme from Pigeon Liver,2.50,Columba livia
5,4dtt,Crystal structure of human insulin degrading e...,3.22,Homo sapiens
6,4dwk,Structure of cystein free insulin degrading en...,3.00,Homo sapiens
7,7rze,Insulin Degrading Enzyme pO/pC,3.30,Homo sapiens
8,7rzf,Insulin Degrading Enzyme O/pC,3.40,Homo sapiens
9,3h44,Crystal Structure of Insulin Degrading Enzyme ...,3.00,Homo sapiens


### FULL ETL RUNNER (ONE CLICK PIPELINE)

In [20]:
def run_etl(query="enzyme", rows=200):
    print("Starting ETL Pipeline...")

    raw = fetch_pdbe_data(query, rows)
    print("Extracted:", len(raw))

    df = transform_data(raw)
    df = clean_data(df)
    print("Transformed:", len(df))

    create_table(conn)

    clear_table(conn)  

    load_data(df, conn)
    create_indexes(conn)

    print("ETL Completed Successfully!")

In [21]:
run_etl("protein", 200)

Starting ETL Pipeline...
Extracted: 200
Transformed: 4


NameError: name 'clear_table' is not defined